# SolarView — train one DeepLabV3 model
Fresh run: choose a Colab GPU and run cells in order. Upload train.py, model.py and CNN.zip only. Training runs on Colab, not the laptop.
Reference split: 80 training, 21 validation, 21 test. Local draft masks are not training labels. Outputs are cleared in this reusable notebook; recorded experiment results are stored separately in results/.


In [ ]:
import sys, subprocess, torch, torchvision
print('Torch',torch.__version__,'Torchvision',torchvision.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print(torch.cuda.get_device_name(0))
subprocess.check_call([sys.executable,'-m','pip','install','-q','Pillow>=10','numpy>=1.26','pandas>=2','matplotlib'])

## 1. Upload train.py, model.py and CNN.zip
Select the two Python files from the kithmini root and the original reference CNN.zip. Solar_Training_UI.zip is no longer needed.


In [ ]:
from google.colab import files
from pathlib import Path
import sys
import io
import zipfile

# 1. Upload the training scripts and dataset
print("Select train.py, model.py and CNN.zip")
uploaded = files.upload()

required_files = {"train.py", "model.py", "CNN.zip"}
missing_files = required_files - set(uploaded)

if missing_files:
    raise ValueError(f"Missing files: {sorted(missing_files)}")

# 2. Create the workspace
BASE = Path("/content/solar_work")
PROJECT = BASE / "code"
DATASET_DIR = BASE / "dataset"

PROJECT.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# 3. Save the Python scripts
for filename in ("train.py", "model.py"):
    (PROJECT / filename).write_bytes(uploaded[filename])

# 4. Extract the dataset
with zipfile.ZipFile(io.BytesIO(uploaded["CNN.zip"])) as archive:
    for entry in archive.infolist():
        target = (DATASET_DIR / entry.filename).resolve()

        if not target.is_relative_to(DATASET_DIR.resolve()):
            raise ValueError("Unsafe ZIP path")

    archive.extractall(DATASET_DIR)

# 5. Find the dataset root
dataset_roots = [
    folder.parent
    for folder in DATASET_DIR.rglob("Images")
    if (folder / "train").is_dir()
    and (folder.parent / "Masks" / "train").is_dir()
]

if len(dataset_roots) != 1:
    raise ValueError(
        "Expected one dataset containing Images/train and Masks/train."
    )

DATA = dataset_roots[0]

# 6. Import the training code and check the dataset
sys.path.insert(0, str(PROJECT))

# Reload the uploaded code if this cell is run again.
for module_name in ('train', 'model'):
    sys.modules.pop(module_name, None)
from train import check_splits

print("Project:", PROJECT)
print("Dataset:", DATA)
print("Image pairs:", check_splits(DATA))

del uploaded


## 2. Check image–mask pairs
White is obstacle; black is sky. Outside the dataset circle is ignored by loss and metrics.

In [ ]:
from train import SkyDataset
from PIL import Image
import matplotlib.pyplot as plt
ds=SkyDataset(DATA,'train',512)
fig,axes=plt.subplots(3,2,figsize=(9,12))
for i,(p,m) in enumerate(ds.pairs[:3]):
    axes[i,0].imshow(Image.open(p));axes[i,0].set_title(p.name)
    axes[i,1].imshow(Image.open(m).convert('L'),cmap='gray',vmin=0,vmax=255)
    for ax in axes[i]:ax.axis('off')
plt.tight_layout()

## 3. Choose a checkpoint folder
USE_DRIVE=True asks you to connect your own Google Drive and saves there. If False, outputs are temporary: download them before closing Colab.

For interrupted runs, set RESUME=True and replace RUN_NAME with the exact previous name. Keep image size unchanged. Resume restores the optimizer but not the exact random-number sequence.

In [ ]:
from datetime import datetime
USE_DRIVE=False
RESUME=False
RUN_NAME=datetime.now().strftime('run_%Y%m%d_%H%M%S')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT=Path('/content/drive/MyDrive/SolarResearch')/RUN_NAME
else:
    OUT=Path('/content/solar_runs')/RUN_NAME
OUT.mkdir(parents=True,exist_ok=True)
print('SAVE THIS PATH:',OUT)

## 4. Train
512 px, batch 2, maximum 40 epochs, early stop after 8 epochs without validation improvement. These are starting settings, not a promised accuracy. If GPU memory is insufficient, start a new run with SIZE=384 and BATCH=2.

General pretrained DeepLabV3 weights are downloaded for initialization. Two binary output heads are then learned. Best checkpoint selection uses validation mIoU only.

In [ ]:
SIZE=512
BATCH=2
EPOCHS=40
cmd=[sys.executable,str(PROJECT/'train.py'),'--data',str(DATA),'--out',str(OUT),'--size',str(SIZE),'--batch',str(BATCH),'--epochs',str(EPOCHS),'--patience','8','--workers','2']
if RESUME:cmd.append('--resume')
subprocess.run(cmd,check=True)

## 5. Review learning curves
These are validation metrics. Do not tune with the final test set.

In [ ]:
import pandas as pd
history=pd.read_csv(OUT/'history.csv')
display(history.tail())
fig,axes=plt.subplots(1,2,figsize=(12,4))
history.plot(x='epoch',y='train_loss',ax=axes[0])
history.plot(x='epoch',y=['miou','sky_iou','obstacle_iou'],ax=axes[1])
plt.show()

## 6. Final test — after freezing your choices
Run once for the selected model. These metrics measure segmentation on the paper dataset at training resolution inside the fisheye disk. They do not measure electricity generation or local-site accuracy.

In [ ]:
subprocess.run([sys.executable,str(PROJECT/'train.py'),'--data',str(DATA),'--out',str(OUT),'--evaluate','--batch',str(BATCH),'--workers','2'],check=True)
import json
display(json.loads((OUT/'test_metrics.json').read_text())['test_metrics'])

## 7. Download a complete backup
This ZIP includes best_model.pt, last_model.pt, history, config and the test_metrics.json just generated. Download before the runtime ends. Put best_model.pt in kithmini/models/. Store the other run files in a new results/deeplabv3/<run-name>/ folder.


In [ ]:
from google.colab import files
import shutil
backup_zip = shutil.make_archive("/content/Solar_Training_Backup", "zip", root_dir=str(OUT))
files.download(backup_zip)


## 8. Optional: preview one new photo
Predictions are not reviewed ground truth. Do not tune the model repeatedly with reserved local test photos.


In [ ]:
from model import load_model,predict,overlay
photos=files.upload()
photo_name=next(n for n in photos if n.lower().endswith(('.jpg','.jpeg','.png')))
photo=Image.open(photo_name).convert('RGB')
trained,info=load_model(OUT/'best_model.pt','cuda')
mask,_=predict(trained,photo,info['size'],'cuda')
fig,axes=plt.subplots(1,3,figsize=(15,5))
for ax,im,title in zip(axes,[photo,mask,overlay(photo,mask)],['Original','Prediction','Obstacle overlay']):
    ax.imshow(im,cmap='gray' if title=='Prediction' else None);ax.set_title(title);ax.axis('off')
plt.show()
mask_path=Path('/content')/(Path(photo_name).stem+'_predicted.png')
mask.save(mask_path);files.download(str(mask_path))